# Oefening 23.1 - Een hybrid-searchscore ontwerpen en testen

**Niveau:** Gemiddeld tot gevorderd  
**Geschatte tijd:** 40 tot 60 minuten  
**Benodigde packages:** pandas

## Doel

Semantische, lexicale en metadata-signalen gecontroleerd combineren.

Inzicht krijgen in het verschil tussen ophalen en reranken.

## Opdracht

Maak vijf zoekresultaten met drie deelscores. Normaliseer de scores en bereken een gewogen hybrid score. Sorteer de resultaten.



Werk stap voor stap:


1. Maak een kleine en controleerbare documentverzameling.

2. Splits retrieval, selectie van context en antwoordgeneratie in aparte stappen.

3. Voer de zoek- of chunkingmethode uit en toon welke bronnen worden geselecteerd.

4. Controleer het antwoord tegen de bron en test ook een vraag zonder passend document.

5. Leg uit of een fout uit retrieval of uit generatie ontstaat.

Hint: Houd bron-ID, score en geselecteerde tekst zichtbaar. Anders is later niet te bepalen waarom een antwoord ontstond.

Bereken hoeveel cross-encoderbeoordelingen nodig zijn voor 100, 100.000 en 10 miljoen documenten. Vergelijk dit met reranking van alleen top-100 resultaten.



Werk stap voor stap:


1. Bereken hoeveel document-queryparen een cross-encoder moet beoordelen bij 100, 100.000 en 10 miljoen documenten.

2. Bereken daarna dezelfde aantallen wanneer eerst retrieval plaatsvindt en alleen de top-100 wordt gererankt.

3. Zet beide scenario's in een tabel en bereken de reductiefactor.

4. Voeg een eenvoudige aanname voor gemiddelde beoordelingstijd of kosten per paar toe.

5. Leg uit waarom retrieval gevolgd door reranking schaalbaarder is dan cross-encoding van de volledige collectie.

Hint: Scheid kandidaatselectie en reranking. De kracht van een cross-encoder zit in nauwkeurige beoordeling van een beperkte kandidatenlijst.

## Werkwijze

1. Probeer de opdracht eerst zelf.
2. Voer de code uit en controleer de tussenresultaten.
3. Voeg minimaal één randgeval toe.
4. Open pas daarna `uitwerking.py` en `uitwerking.md`.


In [ ]:
import pandas as pd

resultaten = pd.DataFrame([
    {"document": "A", "semantisch": 0.88, "lexicaal": 0.20, "actualiteit": 0.90},
    {"document": "B", "semantisch": 0.72, "lexicaal": 1.00, "actualiteit": 0.70},
    {"document": "C", "semantisch": 0.81, "lexicaal": 0.40, "actualiteit": 0.30},
    {"document": "D", "semantisch": 0.55, "lexicaal": 0.80, "actualiteit": 1.00},
    {"document": "E", "semantisch": 0.60, "lexicaal": 0.10, "actualiteit": 0.60},
])

gewichten = {"semantisch": 0.6, "lexicaal": 0.3, "actualiteit": 0.1}
resultaten["hybrid"] = sum(resultaten[k] * w for k, w in gewichten.items())
resultaten = resultaten.sort_values("hybrid", ascending=False)

print(resultaten.round(3).to_string(index=False))

collecties = [100, 100_000, 10_000_000]
vragen_per_dag = 5_000
top_k = 100
milliseconden_per_paar = 4

for documenten in collecties:
    volledig_per_dag = documenten * vragen_per_dag
    rerank_per_dag = top_k * vragen_per_dag
    tijd_volledig_uur = volledig_per_dag * milliseconden_per_paar / 1000 / 3600
    tijd_rerank_uur = rerank_per_dag * milliseconden_per_paar / 1000 / 3600
    print(
        f"Documenten={documenten:,}: volledig={tijd_volledig_uur:,.1f} uur, "
        f"top-{top_k} rerank={tijd_rerank_uur:,.1f} uur"
    )


# Uitwerking 23.1

## Hoe werkt de code?

De drie scores liggen al tussen 0 en 1. In echte systemen moeten scores van verschillende zoekmachines vaak eerst worden genormaliseerd. Actualiteit is alleen nuttig wanneer recentere documenten ook inhoudelijk de voorkeur verdienen. Een verouderd maar formeel geldend beleid mag niet automatisch lager eindigen.

Een cross-encoder leest vraag en document samen en is nauwkeurig, maar moet voor ieder paar worden uitgevoerd. Bij miljoenen documenten is dat te duur en te traag. Daarom haalt een snelle bi-encoder of zoekmachine eerst een beperkte kandidatenlijst op. De cross-encoder rerankt alleen die top-k.


## Verwachte uitvoer

De tabel is gesorteerd op de gewogen hybrid score.

De volledige vergelijking groeit lineair met het aantal documenten. De top-100 rerank blijft voor iedere collectiegrootte gelijk.


## Controleer je uitkomst

- Gewichten tellen op tot 1.
- Scoreberekening is reproduceerbaar.
- Geldigheid en autorisatie zijn harde filters, geen zachte bonus.
- Latency en throughput worden beide gemeten.
- Top-k bevat voldoende recall.
- Reranking heeft time-outs en fallbackgedrag.

## Verdiepingsopdracht

Voer een gevoeligheidsanalyse uit voor semantisch gewicht van 0,3 tot 0,8.

Niveau: Gemiddeld tot gevorderd  |  Geschatte tijd: 40 tot 60 minuten

Bereken hoeveel parallelle workers nodig zijn om de top-100 rerank binnen 300 milliseconden te voltooien.


## Zelf verder testen

Verander minimaal één invoerwaarde en voeg een randgeval toe. Controleer daarna of de conclusie nog steeds door de uitvoer wordt ondersteund.
